In [2]:
import os
import pickle
import logging
import numpy as np
import mdtraj as md
import parmed as pmd
from scipy.spatial.distance import pdist


class FlushFileHandler(logging.FileHandler):
    def emit(self, record):
        super().emit(record)
        self.flush()


logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

# -------------------- Loading the trajectory and topology files --------------------
t = 300
ints = list(range(64, 151, 2))
distances_in_nm = [f"{d/100:.2f}" for d in ints]

radius = 3.5        # A
bin_width = 0.06    # kcal/mol
chunk_size = 500    # frames read at a time

for d in distances_in_nm:

    input_path = f'/gibbs/arghavan/plate_simulations/{t}K/{d}/'
    output_path = f'/gibbs/arghavan/hp-results-pc/pairwise_energy_results/{t}K/{d}/'
    mean_nbr_path = f'/gibbs/arghavan/counting_hbonds_between_graphite_walls/hbond_outputs/{t}K/{d}/all_frame_mean_neighbours.pkl'

    prob_file = f'{output_path}prob_{d}.dat'
    numDens_file = f'{output_path}numDens_{d}.dat'
    dx_file = f'{output_path}pairwise_E_{d}.dx'
    logfile = f'{output_path}log_{d}.log'

    traj_path = f'{input_path}{t}K_{d}.nc'
    top_path = f'{input_path}topol_edited.prmtop'

    # --------------------------- Logging Setup ---------------------------
    os.makedirs(os.path.dirname(logfile), exist_ok=True)
    for h in list(logger.handlers):          # close previous separation's log file
        h.close()
        logger.removeHandler(h)
    file_handler = FlushFileHandler(logfile, mode='w')
    file_handler.setFormatter(logging.Formatter('%(asctime)s %(message)s', datefmt='%I:%M:%S %p'))
    logger.addHandler(file_handler)

    tpl = md.load_topology(top_path)
    parm = pmd.load_file(top_path)

    all_ox_tpl_indices = tpl.select("name O")
    walls = tpl.select("resname =~ 'WALL'")

    # --------------------------- Constants ---------------------------
    Acoeff = float(parm.parm_data['LENNARD_JONES_ACOEF'][0])
    Bcoeff = float(parm.parm_data['LENNARD_JONES_BCOEF'][0])
    chg_lst = np.array(parm.parm_data['CHARGE'][:4], dtype=float) * 18.2223

    o_charge = chg_lst[3]
    h_charge = chg_lst[1]

    # charge products for sites (H1, H2, M) x (H1, H2, M) -- same 9 terms as before
    site_chg = np.array([h_charge, h_charge, o_charge])
    qq = np.outer(site_chg, site_chg)

    float_d = float(d)
    x_center = 2.50
    x_min, x_max = np.float32(x_center - float_d/2), np.float32(x_center + float_d/2)

    pairwise_energies = []
    num_wat_in_box = []
    n_frames = 0
    y_min = None

    for chunk in md.iterload(traj_path, top=tpl, chunk=chunk_size):

        # --------------------------- Surface Between the Plates (frame 0) ---------------------------
        if y_min is None:
            wall_coords = chunk.xyz[0][walls]
            y_min, y_max = np.min(wall_coords[:, 1]), np.max(wall_coords[:, 1])
            z_min, z_max = np.min(wall_coords[:, 2]), np.max(wall_coords[:, 2])

        for xyz in chunk.xyz:
            n_frames += 1
            if n_frames % 100 == 0:
                logger.info(f"Processing frame {n_frames}")

            # ------------------------------ Masking ---------------------------------------
            ox = xyz[all_ox_tpl_indices]
            mask = ((ox[:, 0] >= x_min) & (ox[:, 0] <= x_max) &
                    (ox[:, 1] >= y_min) & (ox[:, 1] <= y_max) &
                    (ox[:, 2] >= z_min) & (ox[:, 2] <= z_max))
            selected_ox_tpl_indices = all_ox_tpl_indices[mask]
            n_wat = len(selected_ox_tpl_indices)
            num_wat_in_box.append(n_wat)
            if n_wat < 2:
                continue

            # (n_wat, 4, 3) coordinates in A; site order O, H1, H2, M
            wat = xyz[selected_ox_tpl_indices[:, None] + np.arange(4)] * 10.0

            # ------------------------------ O-O distances, i < j ---------------------------------------
            r_oo = pdist(wat[:, 0, :])
            ii, jj = np.triu_indices(n_wat, k=1)
            keep = r_oo <= radius
            if not keep.any():
                continue
            ii, jj, r_oo = ii[keep], jj[keep], r_oo[keep]

            # ------------------------------ Pairwise Energy Calculation ---------------------------------------
            lj_energy = Acoeff / r_oo**12 - Bcoeff / r_oo**6
            diff = wat[ii, 1:, None, :] - wat[jj, None, 1:, :]
            electrostatic_energy = np.sum(qq / np.linalg.norm(diff, axis=-1), axis=(1, 2))

            pairwise_energies.append((lj_energy + electrostatic_energy) / 2)

    pairwise_energies = np.concatenate(pairwise_energies) if pairwise_energies else np.empty(0)
    avg_num_wat_in_box = np.mean(num_wat_in_box)

    header = f"[gibbs input] Pairwise energy for ~ {avg_num_wat_in_box:.2f} waters: {n_frames} frames for {t}K - {float(d)*10} A simulation \nColumn: E_n (kcal/mol)"
    np.savetxt(dx_file, pairwise_energies, fmt="%.6f", header=header, delimiter=" ")
    logger.info(f"Saved in {dx_file}")
    logger.info(f"{n_frames} frames - plates_energies.py - plates E + Plot ")
    logger.info(f"{len(pairwise_energies)} pairwise energies calculated.")
    logger.info(f"{avg_num_wat_in_box:.2f} average No. of waters in the box.")

    # --------------------------- Data for plotting ---------------------------
    mean_nbr = []
    with open(mean_nbr_path, 'rb') as f:
        while True:
            try:
                mean_nbr.append(pickle.load(f))
            except EOFError:
                break
    N_nbr = mean_nbr[0][0]

    bins = np.arange(np.floor(np.min(pairwise_energies) / bin_width) * bin_width,
                     np.ceil(np.max(pairwise_energies) / bin_width) * bin_width + bin_width, bin_width)

    counts, edges = np.histogram(pairwise_energies, bins=bins)
    bin_centers = edges[:-1] + bin_width / 2

    prob = counts / (bin_width * np.sum(counts))
    rho_i = prob * N_nbr

    np.savetxt(prob_file, np.column_stack([bin_centers, prob]), fmt=["%.4f", "%.6f"])
    np.savetxt(numDens_file, np.column_stack([bin_centers, rho_i]), fmt=["%.4f", "%.6f"])

# average pairwise energy for each distance

In [4]:
import numpy as np

t = 300
ints = list(range(64, 151, 2))
distances_in_nm = [f"{d/100:.2f}" for d in ints]

out_file = f'/gibbs/arghavan/hp-results-pc/pairwise_energy_results/{t}K/mean_pairwise_E_vs_distance.dat'

rows = []
for d in distances_in_nm:
    energies = np.loadtxt(f'/gibbs/arghavan/hp-results-pc/pairwise_energy_results/{t}K/{d}/pairwise_E_{d}.dx')
    mean_pair_energy = np.mean(energies)
    rows.append((float(d) * 10, mean_pair_energy))   # distance in A, mean E in kcal/mol

rows = np.array(rows)

header = "distance_A  mean_pairwise_E_kcal_per_mol"
np.savetxt(out_file, rows, fmt=["%.1f", "%.4f"], header=header)

In [ ]:
import mdtraj as md
import parmed as pmd
from warnings import filterwarnings
filterwarnings("ignore")


# traj_file = '/Users/arghavan/Graduate Center Dropbox/Arghavan Vedadi Gargari/MyFiles/300K/tip4p2005/md.nc'
top_file = '/Users/arghavan/Graduate Center Dropbox/Arghavan Vedadi Gargari/MyFiles/300K/tip4p2005/topol.prmtop'

# traj = md.load(traj_file, top=top_file)
structure = pmd.load_file(top_file)

/Users/arghavan/bin/miniconda3/envs/lab/lib/python3.10/site-packages/parmed/topologyobjects.py:531: RuntimeWarning: Cannot close a netcdf_file opened with mmap=True, when netcdf_variables or arrays referring to its data still exist. All data arrays obtained from such files refer directly to data on disk, and must be copied before the file can be cleanly closed. (See netcdf_file docstring for more information on mmap.)
  self._tortor_partners = []


In [4]:
structure

<AmberParm 26444 atoms [6611 EPs]; 6611 residues; 19833 bonds; PBC (orthogonal); parameterized>

In [5]:
structure.box

array([49.8513, 49.8513, 79.7621, 90.    , 90.    , 90.    ])

In [ ]:
structure.residues

In [13]:
volume = structure.box[0] * structure.box[1] * structure.box[2]
num_water_molecules = len([atom for atom in structure.atoms if atom.residue.name == 'SOL'])
water_density = num_water_molecules / volume  # molecules per cubic angstrom
print(f"Water Density: {water_density} molecules/Å³")


Water Density: 0.13340668498224467 molecules/Å³
